In [1]:
import pandas as pd
import os

score_file="output/score.sc"
out_file="output/rosetta_score_compare.csv"

# 读取Rosetta score文件
with open(score_file) as f:
    lines=f.readlines()

# 找header
header=None
data=[]
for line in lines:
    if line.startswith("SCORE:"):
        cols=line.strip().split()
        if "total_score" in cols:
            header=cols[1:]
        elif header:
            data.append(cols[1:])

df=pd.DataFrame(data,columns=header)

# 转换数值列
for c in df.columns:
    if c not in ["description"]:
        df[c]=pd.to_numeric(df[c],errors="coerce")

# 保留主要指标
cols=[
    "description",
    "total_score",
    "fa_atr",
    "fa_rep",
    "fa_sol",
    "fa_elec",
    "hbond_bb_sc",
    "hbond_sc",
    "rama_prepro",
    "ref"
]

df=df[cols]

# 添加类别
def classify(x):
    if "ASR" in x:
        return "ASR"
    elif "PLA_top1" in x:
        return "PLA_design"
    elif "WIL" in x:
        return "WT"
    else:
        return "Other"

df["group"]=df["description"].apply(classify)

# 排序
df=df.sort_values(
    by="total_score",
    ascending=True
)

# 保存
df.to_csv(out_file,index=False)

print(df.to_string(index=False))
print("\nSaved:",out_file)

  description  total_score    fa_atr  fa_rep   fa_sol  fa_elec  hbond_bb_sc  hbond_sc  rama_prepro     ref      group
PLA_top1_0001    -1417.356 -2530.194 220.562 1378.630 -603.075      -85.837   -52.170       -5.559 156.476 PLA_design
PLA_top1_0005    -1416.954 -2527.331 223.694 1371.611 -603.165      -85.913   -54.465       -6.192 156.476 PLA_design
PLA_top1_0002    -1415.996 -2543.238 225.864 1381.091 -597.130      -77.298   -56.870       -6.819 156.476 PLA_design
PLA_top1_0004    -1411.914 -2541.499 224.991 1379.747 -602.283      -82.679   -55.774       -5.709 156.476 PLA_design
PLA_top1_0003    -1407.904 -2518.329 219.059 1364.416 -590.155      -77.907   -57.156       -9.349 156.476 PLA_design

Saved: output/rosetta_score_compare.csv


In [2]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

BASE="."
OUTDIR="rosetta_compare"
os.makedirs(OUTDIR,exist_ok=True)

score_files=[]

# 单独score文件
score_files.extend(glob.glob("score/*.sc"))

# output总score
if os.path.exists("output/score.sc"):
    score_files.append("output/score.sc")


def read_rosetta_score(file):
    rows=[]
    header=None
    
    with open(file) as f:
        for line in f:
            if not line.startswith("SCORE:"):
                continue
            
            items=line.strip().split()
            
            if "total_score" in items:
                header=items[1:]
            elif header:
                rows.append(items[1:])
    
    if header is None:
        return pd.DataFrame()
    
    df=pd.DataFrame(rows,columns=header)
    
    return df


all_df=[]

for f in score_files:
    df=read_rosetta_score(f)
    
    if len(df)==0:
        continue
    
    df["source"]=os.path.basename(f)
    all_df.append(df)


data=pd.concat(all_df,ignore_index=True)


# 数值转换
for c in data.columns:
    if c not in ["description","source"]:
        data[c]=pd.to_numeric(data[c],errors="coerce")


# 分组
def group_name(x):
    x=x.upper()
    
    if "ASR" in x:
        return "ASR"
    elif "WIL" in x:
        return "WT"
    elif "PLA_TOP1" in x:
        return "GA"
    else:
        return "Other"


data["group"]=data["description"].apply(group_name)


# 删除未知
data=data[data.group!="Other"]


# 保存
cols=[
    "description",
    "group",
    "total_score",
    "fa_atr",
    "fa_rep",
    "fa_sol",
    "hbond_sc",
    "rama_prepro"
]


data[cols].sort_values(
    "total_score"
).to_csv(
    f"{OUTDIR}/rosetta_energy_summary.csv",
    index=False
)


print(data[cols].sort_values("total_score"))


# ========================
# 绘图
# ========================

metrics=[
    "total_score",
    "fa_rep",
    "fa_atr",
    "fa_sol",
    "hbond_sc",
    "rama_prepro"
]


for metric in metrics:
    
    plt.figure(figsize=(6,5))
    
    groups=["WT","ASR","GA"]
    
    values=[
        data[data.group==g][metric].dropna()
        for g in groups
    ]
    
    plt.boxplot(
        values,
        labels=groups
    )
    
    plt.ylabel(metric)
    plt.title(f"Rosetta {metric}")
    
    plt.tight_layout()
    
    plt.savefig(
        f"{OUTDIR}/{metric}.png",
        dpi=300
    )
    
    plt.close()


print("\nFinished.")
print("Output:",OUTDIR)

                     description group  total_score    fa_atr   fa_rep  \
10                 PLA_top1_0001    GA    -1417.356 -2530.194  220.562   
14                 PLA_top1_0005    GA    -1416.954 -2527.331  223.694   
11                 PLA_top1_0002    GA    -1415.996 -2543.238  225.864   
13                 PLA_top1_0004    GA    -1411.914 -2541.499  224.991   
12                 PLA_top1_0003    GA    -1407.904 -2518.329  219.059   
1   ASR.PLA_78_relaxed_0004_0001   ASR     -935.460 -1731.537  175.083   
7   ASR.PLA_78_relaxed_0005_0001   ASR     -932.351 -1734.813  175.409   
6   ASR.PLA_78_relaxed_0002_0001   ASR     -931.121 -1733.105  173.321   
3   ASR.PLA_78_relaxed_0001_0001   ASR     -929.122 -1732.964  174.563   
2   ASR.PLA_78_relaxed_0003_0001   ASR     -929.108 -1730.184  174.376   
8      WIL_PLA_relaxed_0003_0001    WT     -928.467 -1770.257  181.155   
4      WIL_PLA_relaxed_0005_0001    WT     -926.383 -1769.658  177.465   
5      WIL_PLA_relaxed_0004_0001    WT

/tmp/ipykernel_133017/1746385262.py:135: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(
/tmp/ipykernel_133017/1746385262.py:135: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(
/tmp/ipykernel_133017/1746385262.py:135: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(
/tmp/ipykernel_133017/1746385262.py:135: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(
/tmp/ipykernel_133017/1746385262.py:135: MatplotlibDeprecationWarning: The 'labels' parameter of box


Finished.
Output: rosetta_compare
